# SignBridge AI — Part 2 (Stretch Goal): Word-Level Sign Recognition
### MediaPipe landmark extraction + baseline model on a small set of WLASL word signs

**Only start this once Notebook 1 (alphabet CNN) is working end-to-end.** This is the stretch
piece — if you're short on time, even 5-8 words done well beats 25 done badly.

**Run order:** Runtime → Change runtime type → GPU, then run top to bottom.

This notebook:
1. Pulls `risangbaskoro/wlasl-processed` from Kaggle
2. Picks a small, well-represented subset of word glosses
3. Extracts MediaPipe hand landmarks from each video (63-dim per frame)
4. Trains a baseline classical model (Random Forest) — matches the "classical vs deep" comparison in your proposal
5. Optionally trains a simple LSTM if time allows


## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/SignBridgeAI/models', exist_ok=True)

In [ ]:
!pip install -q kaggle mediapipe opencv-python-headless

from google.colab import files
print("Upload your kaggle.json file:")
uploaded = files.upload()

import os
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d risangbaskoro/wlasl-processed -p /content/data/wlasl --unzip
!ls /content/data/wlasl

## 2. Inspect the metadata and pick your word subset

The WLASL processed mirror includes a JSON file mapping each gloss (word) to its video instances.
Explore it first so you pick glosses that actually have enough samples — don't just pick words
because they're common; pick words WLASL actually has good coverage for.


In [ ]:
import json
import glob

# Locate the metadata json (filename can vary slightly by mirror version — check what's there)
json_candidates = glob.glob('/content/data/wlasl/**/*.json', recursive=True)
print(json_candidates)

In [ ]:
# Adjust this path once you see the actual filename from the cell above
META_PATH = json_candidates[0]

with open(META_PATH) as f:
    meta = json.load(f)

print("Total glosses:", len(meta))

# Rank glosses by number of available instances so you pick well-represented words
counts = [(entry['gloss'], len(entry['instances'])) for entry in meta]
counts.sort(key=lambda x: -x[1])
counts[:40]

In [ ]:
# EDIT THIS: choose 5-10 words to start (expand later if time allows)
TARGET_WORDS = ['hello', 'thankyou', 'yes', 'no', 'please', 'help', 'sorry', 'good']

# Keep only glosses that are both in your target list AND have enough instances
selected = [entry for entry in meta if entry['gloss'] in TARGET_WORDS]
for entry in selected:
    print(entry['gloss'], '->', len(entry['instances']), 'instances')

## 3. Extract MediaPipe hand landmarks per video

For each video: run MediaPipe Hands over every frame, grab 21 landmarks x (x,y,z) = 63 values,
normalize relative to the wrist so it's translation-invariant, and pad/truncate to a fixed
sequence length.

**Note:** video file paths inside the WLASL mirror can be nested oddly — check the actual
structure with `!find /content/data/wlasl -name "*.mp4" | head` if the path logic below doesn't
find your files, and adjust `VIDEO_DIR` accordingly.


In [ ]:
!find /content/data/wlasl -name "*.mp4" | head -5

In [ ]:
import cv2
import mediapipe as mp
import numpy as np

mp_hands = mp.solutions.hands

MAX_FRAMES = 30  # fixed sequence length — pad/truncate to this

def extract_landmarks_from_video(video_path, max_frames=MAX_FRAMES):
    cap = cv2.VideoCapture(video_path)
    sequence = []

    with mp_hands.Hands(static_image_mode=False, max_num_hands=1, min_detection_confidence=0.5) as hands:
        while cap.isOpened() and len(sequence) < max_frames:
            ret, frame = cap.read()
            if not ret:
                break

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = hands.process(rgb)

            if results.multi_hand_landmarks:
                lm = results.multi_hand_landmarks[0]
                coords = np.array([[p.x, p.y, p.z] for p in lm.landmark])
                # Normalize relative to wrist (landmark 0) so position in frame doesn't matter
                coords -= coords[0]
                sequence.append(coords.flatten())
            else:
                sequence.append(np.zeros(63))

    cap.release()

    sequence = np.array(sequence)
    # Pad or truncate to fixed length
    if len(sequence) < max_frames:
        pad = np.zeros((max_frames - len(sequence), 63))
        sequence = np.vstack([sequence, pad]) if len(sequence) > 0 else pad
    else:
        sequence = sequence[:max_frames]

    return sequence

In [ ]:
# Adjust VIDEO_DIR based on what the `find` command above showed you
VIDEO_DIR = '/content/data/wlasl/videos'  # <-- CHECK AND EDIT THIS

X_sequences = []
y_labels = []

for entry in selected:
    gloss = entry['gloss']
    for inst in entry['instances']:
        video_id = inst['video_id']
        video_path = f"{VIDEO_DIR}/{video_id}.mp4"
        if not os.path.exists(video_path):
            continue
        seq = extract_landmarks_from_video(video_path)
        X_sequences.append(seq)
        y_labels.append(gloss)

print(f"Extracted {len(X_sequences)} sequences across {len(set(y_labels))} glosses")

In [ ]:
import pickle

X_sequences = np.array(X_sequences)
y_labels = np.array(y_labels)

# Save extracted features so you don't need to re-run video processing if runtime disconnects
np.save('/content/drive/MyDrive/SignBridgeAI/X_sequences.npy', X_sequences)
with open('/content/drive/MyDrive/SignBridgeAI/y_labels.pkl', 'wb') as f:
    pickle.dump(y_labels, f)

print(X_sequences.shape)

## 4. Baseline: classical ML on flattened landmark sequences

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score

# Flatten sequences for classical models (they don't model time order, just overall pattern)
X_flat = X_sequences.reshape(X_sequences.shape[0], -1)

le_word = LabelEncoder()
y_encoded = le_word.fit_transform(y_labels)

X_train, X_test, y_train, y_test = train_test_split(
    X_flat, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

print("Random Forest Macro F1:", f1_score(y_test, rf_pred, average='macro'))
print(classification_report(y_test, rf_pred, target_names=le_word.classes_))

## 5. Optional: simple LSTM (only if time allows)

This models the actual temporal order of the sequence, which the flattened classical model above
ignores. Worth doing for the "comparison" deliverable, but don't let this be a blocker — the
Random Forest baseline above already gives you a legitimate word-recognition result.


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

num_classes_word = len(le_word.classes_)

X_train_seq, X_test_seq, y_train_seq, y_test_seq = train_test_split(
    X_sequences, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

y_train_cat = to_categorical(y_train_seq, num_classes_word)
y_test_cat = to_categorical(y_test_seq, num_classes_word)

lstm_model = Sequential([
    LSTM(64, input_shape=(MAX_FRAMES, 63), return_sequences=False),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(num_classes_word, activation='softmax')
])

lstm_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
lstm_model.summary()

In [ ]:
history = lstm_model.fit(
    X_train_seq, y_train_cat,
    validation_data=(X_test_seq, y_test_cat),
    epochs=40,
    batch_size=8,
    callbacks=[EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)]
)

In [ ]:
y_pred_lstm = np.argmax(lstm_model.predict(X_test_seq), axis=1)
print("LSTM Macro F1:", f1_score(y_test_seq, y_pred_lstm, average='macro'))
print(classification_report(y_test_seq, y_pred_lstm, target_names=le_word.classes_))

## 6. Save whichever model performed best

In [ ]:
import pickle

# Save Random Forest baseline
with open('/content/drive/MyDrive/SignBridgeAI/models/word_rf_baseline.pkl', 'wb') as f:
    pickle.dump(rf, f)

# Save LSTM if you trained it
lstm_model.save('/content/drive/MyDrive/SignBridgeAI/models/word_lstm.keras')

with open('/content/drive/MyDrive/SignBridgeAI/models/word_label_encoder.pkl', 'wb') as f:
    pickle.dump(le_word, f)

print("Saved word-sign models to Google Drive.")

## Next: the Streamlit app
Both this model and the alphabet CNN from Notebook 1 plug into a local Streamlit app
(runs on your own machine, not Colab, since it needs live webcam access). Ask for that script
once your models are trained and saved.
